In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import seaborn as sns

In [ ]:
df=pd.read_csv("world_bank_country_data.csv")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

In [ ]:
#Filtramos a nuestro pais
df= df[df["country"] == "Netherlands"]
df
#Vemos que hay unos cuantos outliers

In [ ]:
#Detectamos outliers conel modelo del iqr dado en gestion
q1 = df['value'].quantile(0.25)
q3 = df['value'].quantile(0.75)
iqr = q3 - q1
print(iqr)
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

In [ ]:
outliers_iqr = (df['value'] < limite_inferior)|(df['value'] > limite_superior)
df[outliers_iqr]

In [ ]:
outliers_iqr.sum()
#De 1600 filas hay 300 que son outliers, que es bastante, pero al ser datos economicos no vamos a imputarlos porque es algo normal

In [ ]:
df_filtrado = df
df_filtrado

In [ ]:
puerto = pd.read_csv("datosPuerto.csv", decimal=",")
#cargamos y iltramos el archivo del puerto. Pasamos la fecha a datetime y extraemos el año para agrupar los kilos por año
puerto['FECHA'] = pd.to_datetime(puerto['FECHA'])
puerto['FECHA'] = puerto['FECHA'].dt.year
#Filtramos las exportaciones y importaciones
exportaciones = puerto[(puerto["PAIS_FINAL"] == "NL") & (puerto["PAIS_ORIGEN"] == "ES")]
importaciones = puerto[(puerto["PAIS_FINAL"] == "ES") & (puerto["PAIS_ORIGEN"] == "NL")]
#Agrupamos los kilos por años
exportaciones = exportaciones.groupby("FECHA")["PESO_KILOS"].sum()
importaciones = importaciones.groupby("FECHA")["PESO_KILOS"].sum()

print("Importaciones", importaciones)

In [ ]:
exportaciones = {
2009: 366543925,
2010: 410957443,
2011: 754842617,
2012: 540914746,
2013: 851701657,
2014: 940298629,
2015: 595610110,
2016: 844578531,
2017: 951613421,
2018: 971474553,
2019: 1154433286,
2020: 825988465,
2021: 1040936381,
2022: 661942327,
2023: 483180404,
2024: 741719313}


In [ ]:
importaciones = {  
2009: 826407028,  
2010: 1096816180,  
2011: 786733730,  
2012: 925922346,  
2013: 732426144,  
2014: 785162500,  
2015: 790342418,  
2016: 690588712,  
2017: 683978612,  
2018: 696356457,  
2019: 831587812,  
2020: 520076382,  
2021: 655221933,  
2022: 254646377,  
2023: 323879014,  
2024: 376930596}

In [ ]:
#Hacemos una serie con los kilos para que sea mas facil meterlo en el merge.
df_kilos_exportaciones = pd.Series(exportaciones).reset_index()
df_kilos_exportaciones.columns = ["date", "kilos"]
#hacemos un pivot_table y un merge para que se junten los datos de los kilos y los indicadores
df_pivot_exportaciones = df_filtrado.pivot_table(
    index="date",
    columns="indicator",
    values="value").reset_index()
df_merge_exportaciones = df_kilos_exportaciones.merge(df_pivot_exportaciones, on="date", how="left")
#Hacemos la correlacion de los indicadores para ver cuales tienen mas importancia
corr_exportaciones = df_merge_exportaciones.corr(numeric_only=True)
corr_exportaciones["kilos"].sort_values(ascending=False)

In [ ]:
#Filttramos solo los indicadores mas correlacion tienen(poitiva y negativa para que el heatmap sea visible)
indicadores_elegidos = ["Export value index (2015 = 100)", "Import value index (2015 = 100)", "High-technology exports (% of manufactured exports)", "Total electricity output (GWh)", "Net food imports by crop - Barley", "Machinery and transport equipment (% of value added in manufacturing)", "High-technology exports (current US$)", "Food imports (% of merchandise imports)","kilos" ]
corr_exportaciones = corr_exportaciones.loc[indicadores_elegidos, indicadores_elegidos]

In [ ]:
#Hacemos el heatmap
plt.figure(figsize=(20, 20))
sns.heatmap(
    corr_exportaciones,
    annot=True,
    cmap="coolwarm",
    linewidths=0.5,
    fmt=".2f"
)

plt.title("Heatmap de correlaciones entre indicadores y kilos enviados")
plt.tight_layout()
plt.show()

In [ ]:
#Hacemos una serie con los kilos para que sea mas facil meterlo en el merge.
df_kilos_importaciones = pd.Series(importaciones).reset_index()
df_kilos_importaciones.columns = ["date", "kilos"]
#hacemos un pivot_table y un merge para que se junten los datos de los kilos y los indicadores
df_pivot_importaciones = df_filtrado.pivot_table(
    index="date",
    columns="indicator",
    values="value").reset_index()
df_merge_importaciones = df_kilos_importaciones.merge(df_pivot_importaciones, on="date", how="left")
#Hacemos la correlacion de los indicadores para ver cuales tienen mas importancia
corr_importaciones = df_merge_importaciones.corr(numeric_only=True)
corr_importaciones["kilos"].sort_values(ascending=False)

In [ ]:
#Filttramos solo los indicadores mas correlacion tienen(poitiva y negativa para que el heatmap sea visible)
indicadores_elegidos = [ "Export value index (2015 = 100)", "Import value index (2015 = 100)", "High-technology exports (% of manufactured exports)", "Total electricity output (GWh)", "Net food imports by crop - Barley", "Machinery and transport equipment (% of value added in manufacturing)", "High-technology exports (current US$)", "Food imports (% of merchandise imports)","kilos" ]
corr_importaciones = corr_importaciones.loc[indicadores_elegidos, indicadores_elegidos]

In [ ]:
#Hacemos el heatmap
plt.figure(figsize=(20, 20))
sns.heatmap(
    corr_importaciones,
    annot=True,
    cmap="coolwarm",
    linewidths=0.5,
    fmt=".2f"
)

plt.title("Heatmap de correlaciones entre indicadores y kilos enviados")
plt.tight_layout()
plt.show()